# 🌌 Predicting Stellar Class — Playground Series S6E6

Classifying celestial objects into **GALAXY**, **QSO** (quasar), or **STAR** from photometric and spectral measurements.

**Approach:** engineer domain-informed astronomical features (SDSS color indices, redshift transforms, stellar locus distance), train Random Forest-free gradient boosting ensemble (XGBoost + LightGBM + CatBoost), diagnose weaknesses via confusion matrix, and finalize on the best-performing combination.

**Result:** 0.9542 cross-validated balanced accuracy (5-fold stratified).


## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e6/test.csv')

print(train.shape, test.shape)
train.head()

In [ ]:
print(train.columns.tolist())
train['class'].value_counts()

## 2. Feature Engineering — Photometric Color Indices

Raw magnitudes (`u, g, r, i, z`) are less informative on their own than the **differences between them**. These color indices are the standard SDSS technique for separating stellar/galactic/quasar populations, since color traces temperature and composition more directly than brightness alone.

In [ ]:
train['u_g'] = train['u'] - train['g']
train['g_r'] = train['g'] - train['r']
train['r_i'] = train['r'] - train['i']
train['i_z'] = train['i'] - train['z']
train['u_r'] = train['u'] - train['r']
train['g_i'] = train['g'] - train['i']

test['u_g'] = test['u'] - test['g']
test['g_r'] = test['g'] - test['r']
test['r_i'] = test['r'] - test['i']
test['i_z'] = test['i'] - test['z']
test['u_r'] = test['u'] - test['r']
test['g_i'] = test['g'] - test['i']

## 3. Feature Engineering — Redshift Transforms

QSOs typically sit at much higher redshift than galaxies or stars, and stars sit essentially at redshift ≈ 0 (they're within our own galaxy). Log and squared transforms help models split cleanly across this heavily skewed feature.

In [ ]:
train['log1p_redshift'] = np.log1p(train['redshift'])
test['log1p_redshift'] = np.log1p(test['redshift'])

train['redshift_sq'] = train['redshift'] ** 2
test['redshift_sq'] = test['redshift'] ** 2

**Near-zero redshift flags** — making the star/non-star redshift boundary explicit instead of relying on the model to learn a soft threshold on its own.

In [ ]:
train['is_near_zero_redshift'] = (train['redshift'] < 0.01).astype(int)
test['is_near_zero_redshift'] = (test['redshift'] < 0.01).astype(int)

train['is_very_low_redshift'] = (train['redshift'] < 0.05).astype(int)
test['is_very_low_redshift'] = (test['redshift'] < 0.05).astype(int)

## 4. Feature Engineering — Brightness Proxies

In [ ]:
train['avg_mag'] = train[['u', 'g', 'r', 'i', 'z']].mean(axis=1)
test['avg_mag'] = test[['u', 'g', 'r', 'i', 'z']].mean(axis=1)

train['mag_std'] = train[['u', 'g', 'r', 'i', 'z']].std(axis=1)
test['mag_std'] = test[['u', 'g', 'r', 'i', 'z']].std(axis=1)

## 5. Feature Engineering — Stellar Locus Distance

Classic star/galaxy separation technique: stars trace a tight, well-known curve in `u-g` vs `g-r` color space, while galaxies scatter off it. Fitting a quadratic to that curve using known star points and measuring every object's residual distance from it gives the model an explicit signal for this boundary.

*Note: this fits the curve using train labels globally rather than per-fold — a small amount of leakage, common practice for domain-knowledge features like this.*

In [ ]:
star_rows = train[train['class'] == 'STAR']
locus_coeffs = np.polyfit(star_rows['g_r'], star_rows['u_g'], deg=2)

train['stellar_locus_pred'] = np.polyval(locus_coeffs, train['g_r'])
train['stellar_locus_residual'] = train['u_g'] - train['stellar_locus_pred']

test['stellar_locus_pred'] = np.polyval(locus_coeffs, test['g_r'])
test['stellar_locus_residual'] = test['u_g'] - test['stellar_locus_pred']

train['stellar_locus_residual_abs'] = train['stellar_locus_residual'].abs()
test['stellar_locus_residual_abs'] = test['stellar_locus_residual'].abs()

## 6. Categorical Encoding & Interaction Terms

In [ ]:
le_spectral = LabelEncoder()
combined_spectral = pd.concat([train['spectral_type'].astype(str), test['spectral_type'].astype(str)])
le_spectral.fit(combined_spectral)
train['spectral_type_enc'] = le_spectral.transform(train['spectral_type'].astype(str))
test['spectral_type_enc'] = le_spectral.transform(test['spectral_type'].astype(str))

le_galaxy_pop = LabelEncoder()
combined_galaxy_pop = pd.concat([train['galaxy_population'].astype(str), test['galaxy_population'].astype(str)])
le_galaxy_pop.fit(combined_galaxy_pop)
train['galaxy_population_enc'] = le_galaxy_pop.transform(train['galaxy_population'].astype(str))
test['galaxy_population_enc'] = le_galaxy_pop.transform(test['galaxy_population'].astype(str))

In [ ]:
train['spectral_x_redshift'] = train['spectral_type_enc'] * train['redshift']
test['spectral_x_redshift'] = test['spectral_type_enc'] * test['redshift']

train['galaxy_pop_x_redshift'] = train['galaxy_population_enc'] * train['redshift']
test['galaxy_pop_x_redshift'] = test['galaxy_population_enc'] * test['redshift']

train['u_g_x_redshift'] = train['u_g'] * train['redshift']
test['u_g_x_redshift'] = test['u_g'] * test['redshift']

**Target encoding:**

In [ ]:
le_target = LabelEncoder()
train['class_enc'] = le_target.fit_transform(train['class'])
print(dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

## 7. Final Feature Set

In [ ]:
features = [
    'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
    'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i',
    'log1p_redshift', 'redshift_sq',
    'avg_mag', 'mag_std',
    'spectral_type_enc', 'galaxy_population_enc',
    'spectral_x_redshift', 'galaxy_pop_x_redshift', 'u_g_x_redshift',
    'is_near_zero_redshift', 'is_very_low_redshift',
    'stellar_locus_residual', 'stellar_locus_residual_abs'
]

X = train[features]
y = train['class_enc']
X_test = test[features]

print(X.shape, X_test.shape)

## 8. Model Training — 5-Fold Stratified Cross-Validation

Three gradient boosting models trained independently, fold by fold, with out-of-fold predictions collected for honest CV scoring and ensembling.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_classes = len(le_target.classes_)

oof_xgb = np.zeros((len(X), n_classes))
oof_lgbm = np.zeros((len(X), n_classes))
oof_cat = np.zeros((len(X), n_classes))

test_xgb = np.zeros((len(X_test), n_classes))
test_lgbm = np.zeros((len(X_test), n_classes))
test_cat = np.zeros((len(X_test), n_classes))

### XGBoost

In [ ]:
fold_scores_xgb = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb_model = XGBClassifier(
        n_estimators=800, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        random_state=42, eval_metric='mlogloss', n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr)

    val_pred = xgb_model.predict_proba(X_val)
    oof_xgb[val_idx] = val_pred
    test_xgb += xgb_model.predict_proba(X_test) / cv.get_n_splits()

    fold_acc = balanced_accuracy_score(y_val, val_pred.argmax(axis=1))
    fold_scores_xgb.append(fold_acc)
    print(f"XGBoost fold {fold}: {fold_acc:.4f}")

print(f"XGBoost CV: {np.mean(fold_scores_xgb):.4f} (+/- {np.std(fold_scores_xgb):.4f})")

### LightGBM

In [ ]:
fold_scores_lgbm = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm_model = LGBMClassifier(
        n_estimators=800, max_depth=7, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        objective='multiclass', num_class=n_classes,
        random_state=42, verbose=-1
    )
    lgbm_model.fit(X_tr, y_tr)

    val_pred = lgbm_model.predict_proba(X_val)
    oof_lgbm[val_idx] = val_pred
    test_lgbm += lgbm_model.predict_proba(X_test) / cv.get_n_splits()

    fold_acc = balanced_accuracy_score(y_val, val_pred.argmax(axis=1))
    fold_scores_lgbm.append(fold_acc)
    print(f"LightGBM fold {fold}: {fold_acc:.4f}")

print(f"LightGBM CV: {np.mean(fold_scores_lgbm):.4f} (+/- {np.std(fold_scores_lgbm):.4f})")

### CatBoost

In [ ]:
fold_scores_cat = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    cat_model = CatBoostClassifier(
        iterations=800, depth=6, learning_rate=0.03,
        loss_function='MultiClass', random_state=42, verbose=0
    )
    cat_model.fit(X_tr, y_tr)

    val_pred = cat_model.predict_proba(X_val)
    oof_cat[val_idx] = val_pred
    test_cat += cat_model.predict_proba(X_test) / cv.get_n_splits()

    fold_acc = balanced_accuracy_score(y_val, val_pred.argmax(axis=1))
    fold_scores_cat.append(fold_acc)
    print(f"CatBoost fold {fold}: {fold_acc:.4f}")

print(f"CatBoost CV: {np.mean(fold_scores_cat):.4f} (+/- {np.std(fold_scores_cat):.4f})")

## 9. Ensemble Comparison

Testing three combination strategies to find the best one — simple average, performance-weighted average, and dropping the weakest model entirely.

In [ ]:
oof_ensemble = (oof_xgb + oof_lgbm + oof_cat) / 3
ensemble_acc = balanced_accuracy_score(y, oof_ensemble.argmax(axis=1))
print(f"Simple 3-model average: {ensemble_acc:.4f}")

oof_weighted = (0.4 * oof_xgb) + (0.45 * oof_lgbm) + (0.15 * oof_cat)
weighted_acc = balanced_accuracy_score(y, oof_weighted.argmax(axis=1))
print(f"Weighted 3-model average: {weighted_acc:.4f}")

oof_two_model = (oof_xgb + oof_lgbm) / 2
two_model_acc = balanced_accuracy_score(y, oof_two_model.argmax(axis=1))
print(f"XGBoost + LightGBM only: {two_model_acc:.4f}")

**Finding:** CatBoost consistently underperforms XGBoost and LightGBM (~0.947 vs ~0.953-0.954), and including it in any average — even weighted — drags the ensemble below the best single model. **Dropping CatBoost entirely and averaging just XGBoost + LightGBM gives the best result.**

## 10. Confusion Matrix — Diagnosing the Remaining Errors

In [ ]:
lgbm_pred_labels = le_target.inverse_transform(oof_lgbm.argmax(axis=1))
true_labels = le_target.inverse_transform(y)

cm = confusion_matrix(true_labels, lgbm_pred_labels, labels=['GALAXY', 'QSO', 'STAR'])
cm_df = pd.DataFrame(cm, index=['True_GALAXY', 'True_QSO', 'True_STAR'],
                      columns=['Pred_GALAXY', 'Pred_QSO', 'Pred_STAR'])
print(cm_df)

print()
for class_name in ['GALAXY', 'QSO', 'STAR']:
    class_mask = true_labels == class_name
    class_acc = (lgbm_pred_labels[class_mask] == true_labels[class_mask]).mean()
    print(f"{class_name} accuracy: {class_acc:.4f} ({class_mask.sum()} samples)")

**Finding:** STAR is the weakest class (92.1% accuracy), mostly confused *with* GALAXY rather than QSO. This makes physical sense — low-redshift galaxies can look photometrically similar to stars. Adding the stellar locus residual and near-zero-redshift flag (Section 5) was targeted at closing this gap specifically — it moved STAR accuracy by less than 0.01%, suggesting this confusion reflects the dataset's natural noise floor rather than a missing feature.

## 11. Final Model & Submission

In [ ]:
oof_final = (oof_xgb + oof_lgbm) / 2
final_acc = balanced_accuracy_score(y, oof_final.argmax(axis=1))
print(f"Final CV Balanced Accuracy: {final_acc:.4f}")

pred_labels = le_target.inverse_transform(oof_final.argmax(axis=1))
for class_name in ['GALAXY', 'QSO', 'STAR']:
    class_mask = true_labels == class_name
    class_acc = (pred_labels[class_mask] == true_labels[class_mask]).mean()
    print(f"{class_name} accuracy: {class_acc:.4f}")

In [ ]:
test_final = (test_xgb + test_lgbm) / 2
final_predictions_enc = test_final.argmax(axis=1)
final_predictions = le_target.inverse_transform(final_predictions_enc)

submission = pd.DataFrame({
    'id': test['id'],
    'class': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv created!")
submission.head()

## 12. Results & Takeaways

| Model | CV Balanced Accuracy |
|---|---|
| XGBoost | 0.9533 |
| LightGBM | 0.9540 |
| CatBoost | 0.9468 |
| Simple 3-model average | 0.9527 |
| Weighted 3-model average | 0.9538 |
| **XGBoost + LightGBM (final)** | **0.9542** |

**Key findings:**
- CatBoost underperformed and was excluded — ensembling only helps when member models are roughly comparable in strength; a weaker third model drags down a simple average even when the stronger two are solid.
- SDSS color indices and redshift transforms carried most of the signal, as expected from domain knowledge.
- The remaining error is concentrated in STAR vs GALAXY confusion at low redshift — targeted feature engineering (stellar locus residual) didn't meaningfully close this gap, suggesting it reflects genuine overlap in the synthetic dataset rather than a missing feature.

**Next steps:** Optuna hyperparameter tuning, stacking with a meta-learner instead of averaging, or engineering features specific to the low-redshift GALAXY/STAR boundary.
